# Day 3: Batch Ingestion (X12 EDI Claims & Debezium CDC)
This notebook simulates batch parsing of synthetic raw EDI lines and Debezium JSON records.

In [ ]:
!pip install pyspark pyiceberg duckdb splink

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, lit, expr

spark = SparkSession.builder \
    .appName("Day03_Batch_CDC_Simulation") \
    .getOrCreate()

print("Spark session initialized.")

### X12 EDI Batch Parsing Simulation

In [ ]:
raw_edi_data = [
    ("ISA*00*          *00*          *ZZ*1234567890     *ZZ*9876543210     *100101*1200*U*00401*000000001*0*T*:~",),
    ("GS*HC*1234567890*9876543210*20100101*1200*1*X*004010X098A1~",)
]

df_raw_edi = spark.createDataFrame(raw_edi_data, ["value"])

df_parsed_edi = df_raw_edi \
    .withColumn("payload_id", expr("uuid()")) \
    .withColumn("source_system_id", lit("SIM_EDI_X12")) \
    .withColumn("raw_payload_json", col("value")) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .select("payload_id", "source_system_id", "raw_payload_json", "ingestion_timestamp")

df_parsed_edi.show(truncate=False)

### Debezium CDC Processing Simulation

In [ ]:
cdc_data = [
    ('{"before": null, "after": {"id": 1, "status": "active"}, "op": "c"}',),
    ('{"before": {"id": 1, "status": "active"}, "after": {"id": 1, "status": "inactive"}, "op": "u"}',)
]

df_cdc_raw = spark.createDataFrame(cdc_data, ["raw_json"])

df_cdc_processed = df_cdc_raw \
    .withColumn("payload_id", expr("uuid()")) \
    .withColumn("source_system_id", lit("SIM_DEBEZIUM_CDC")) \
    .withColumn("raw_payload_json", col("raw_json")) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .select("payload_id", "source_system_id", "raw_payload_json", "ingestion_timestamp")

df_cdc_processed.show(truncate=False)